# 逻辑回归在故障预警与分类中的应用

## 1. 项目背景
在一些重要的场景下，系统的可靠高效运行对现代社会至关重要。然而，这些系统易受到各种电气故障的影响，这些故障可能会中断设备运行、损坏设备并造成安全隐患。  
早期检测并准确分类这些故障对于最大限度地减少停机时间、确保设备保护并维持系统稳定性至关重要。  
本项目选取机器学习在输电线路电气故障检测和分类中的应用。项目旨在建立一个逻辑回归模型，通过综合分析电压和电流测量值，识别各类故障，例如开路故障、单线接地故障、线间故障和三相故障。 利用机器学习算法构建准确的故障预警分类模型  
通过本次实训，学员将深入理解逻辑回归算法的基本原理和操作。通过构建该模型，可以有效地对故障进行预警分类，为维修决策提供科学依据，提高维修效率，优化维修流程。


## 2. 数据处理

### 2.1 数据预览

|数据总数 |特征总数 | 
|-|-|
|7861 |10 | 

**特征说明**


* G：是否接地（数据类型：整数）
* C：是否接C线（数据类型：整数）
* B：是否接B线（数据类型：整数）
* A：是否接A线（数据类型：整数）
* Ia：A 相电流（数据类型，浮点型）
* Ib：B 相电流（数据类型：浮点型）
* Ic：C 相电流（数据类型：浮点型）
* Va：A 相电压（数据类型：浮点型）
* Vb：B 相电压（数据类型：浮点型）
* Vc：C 相电压（数据类型：浮点型）

### 2.2 导入必要的库

首先，我们会导入我们所需要的所有科学计算库。这是在数据处理与分析前的必要步骤：

In [ ]:
import numpy as np                                 # 导入数值计算库NumPy，提供高效的数组操作和数学函数
import pandas as pd                                # 导入数据分析库Pandas，提供DataFrame等数据结构用于数据处理
import seaborn as sns                              # 导入数据可视化库Seaborn，基于matplotlib提供更高级的统计图表
import matplotlib.pyplot as plt                    # 导入绘图库matplotlib的pyplot模块，提供基本的绘图功能

### 2.3 加载数据集

接下来导入本项目已准备好的数据集，并展示数据集的维度。

In [ ]:
data=pd.read_csv('/home/jovyan/work/datasets/6892c8359e78856c69427a93-momodel/classData.csv')  #确保数据集在同级目录下，如果不在，请自行修改数据集地址。
data.shape # 描述数据集维度

可以看到，一共有7860组数据。  
再将所有错误合并到一个fault_type中。根据G,C,B,A的数值不同，将GCBA值相同的数据统一划分到一起，使用Fault_Type表示，Fault_Type的值即GCBA拼接而成。

In [ ]:
data['Fault_Type'] =data['G'].astype('str') + data['C'].astype('str') + data['B'].astype('str') + data['A'].astype('str') # 集结所有的错误类型
data.head(10) # 显示前十行数据

根据数据类型对错误进行命名：

In [ ]:
data.loc[data['Fault_Type'] == '0000', 'Fault_Type'] = 'Np Fault'
data.loc[data['Fault_Type'] == '1001', 'Fault_Type'] = 'Line A to Ground Fault'
data.loc[data['Fault_Type'] == '0110', 'Fault_Type'] = 'Line B to Line C Fault'
data.loc[data['Fault_Type'] == '1011', 'Fault_Type'] = 'Line A Line B to Ground Fault'
data.loc[data['Fault_Type'] == '0111', 'Fault_Type'] = 'Line A Line B Line C'
data.loc[data['Fault_Type'] == '1111', 'Fault_Type'] = 'Line A Line B Line C to Ground Fault'

### 2.4 数据可视化

接下来进行数据可视化。我们使用matplotlib库进行可视化处理，可以直观地展示数据的分布。  
此外，还可以看到不同数据的数值分布情况。通过这样的方式，我们可以轻松地观察到数据的分布情况，从而更深入地理解的内在特征。

In [ ]:
ax = plt.figure(figsize = (8,8)) # 创建新的图形对象，并设置图形尺寸为8x8
ax = plt.subplot(1,1,1)  # 在图形中创建1x1的子图网络（即单个图表），并选择第一个位置
ax = sns.countplot(x='Fault_Type', data=data) # 使用seaborn绘制柱状图。使用Fault_Type列作为x轴
ax.bar_label(ax.containers[0]) # 在每个柱子上方添加数值标签
plt.title("Fault Type", fontsize=15) # 设置图表标题。分别是标题文本和字体大小         
plt.xticks(rotation=55) # 旋转x轴标签65度，避免长标签重叠
plt.tight_layout() # 自动调整子图参数，使得图形元素不重叠

展现数据数值变化范围：这里是Ia,Ib,Ic的数据数值变化范围

In [ ]:
plt.figure(figsize = (10,4))
plt.plot(data["Ia"])
plt.plot(data["Ib"])
plt.plot(data["Ic"]);

展现数据Va,Vb,Vc的数值变化范围。

In [ ]:
plt.figure(figsize = (10,4))
plt.plot(data["Va"])
plt.plot(data["Vb"])
plt.plot(data["Vc"]);

用数字表示错误类型。可以看到，同一类的错误都已被归为一样的Fault_Type。与之前不同的是，这里的Fault已用数字进行表示，极大提高了效率。

In [ ]:
# 将标签转化为数字类型
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
data['Fault_Type'] = encoder.fit_transform(data['Fault_Type'])
data.head()

接着查看对应关系：  
原始标签到数字的映射关系:  
Line A Line B Line C -> 0  
Line A Line B Line C to Ground Fault -> 1  
Line A Line B to Ground Fault -> 2  
Line A to Ground Fault -> 3  
Line B to Line C Fault -> 4  
Np Fault -> 5  


In [ ]:
print("原始标签到数字的映射关系:")
for i, label in enumerate(encoder.classes_):
    print(f"{label} -> {i}")

### 2.5 模型构建与训练

#### 概念引入：逻辑回归如何操作？

**1. 构建线性模型**  
首先，逻辑回归使用一个线性函数来组合输入特征：
$z = \beta_{0} + \beta_{1} X_{1} + \beta_{2} X_{2} + \beta_{3} X_{3} + ... + \beta_{n} X_{n}$  
$\beta_{0}$为截距（偏置），$\beta_{1},\beta_{2}...\beta_{n}$是特征系数，$X_{1}, X_{2}, ...X_{n}$ 是输入特征  
**2. 应用sigmoid函数**  
将线性模型的输出$z$通过sigmoid函数（也称为logistic函数）转换到(0,1)区间，得到属于正类（通常标记为1）的概率：  
$\sigma(z)=\frac{1}{1+e^{-z}}$  
该函数将任意实数映射到（0,1）区间内，当$z$很大时，$\sigma(z)$趋近于1；当$z$很小时，$、sigma(z)$趋近于0  
**3. 设置阈值**  
设置一个阈值，默认通常为0.5，将概率值转化为类别标签  
**4. 参数估计（训练)**  
逻辑回归使用最大似然估计（MLE）来估计模型参数（即系数$\beta$）。目标是通过调整参数，使得模型预测的概率尽可能接近真实标签。  
**5. 优化算法**  
为了最小化损失函数，通常使用梯度下降法或其他变种来迭代更新参数$\beta$。


---



接下来我们进行模型的构建与训练。首先第一步我们需要必要的包，包括逻辑回归库，用于分离训练集测试集的库，以及等等。  
流程：  
1. 实例化逻辑回归模型  
2. 使用训练数据拟合逻辑回归模型，模型根据X_train和Y_train自动调整参数  
3. 使用测试集X_test查看模型预测精度

In [ ]:
# 导入工具以构建训练集和测试集
from sklearn.model_selection import train_test_split
# 导入必要的包，包括逻辑回归库
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt

# 变量分离：
# X - 特征矩阵：包含所有特征列，但不包含目标列
# Y - 目标向量：只包含Fault_Type列
X = data.drop(['Fault_Type'],axis=1) # 删除目标列作为特征集
y = data['Fault_Type'] # 目标列作为标签集

# 将数据集分成训练集和测试集，这里设置测试集比例为0.2，固定随机种子确保结果可复现
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 逻辑回归模型训练和预测：
# 1. 创建逻辑回归分类器对象，使用默认参数
logreg = LogisticRegression(max_iter=200)
# 2. 使用训练数据拟合模型
logreg.fit(X_train, y_train)
# 3. 使用训练好的模型对测试集进行预测
y_pred_lr = logreg.predict(X_test)


### 2.6 结果与分析

#### 生成分类报告表格 


从中可以看到，训练集准确率（Training Accuracy）和测试集准确率（Model Accuracy Score）接近，说明模型没有明显的过拟合。  
分析下列表格，得到下列结论：  
1.类别不平衡问题，表现最好的类别可以完美识别，表现最差的类别召回率仅25%，原因可能是模型对于少数类识别能力差。  
2.特异性问题：有的类别精确率仅57%，说明有大量误报。此类故障可能会产生假警报。  
3.Line B to Line C Fault类故障可能会被系统忽略，造成安全隐患。

In [ ]:
log_train = round(logreg.score(X_train, y_train) * 100, 2)
log_accuracy = round(accuracy_score(y_pred_lr, y_test) * 100, 2)

print("Training Accuracy    :",log_train ,"%")
print("Model Accuracy Score :",log_accuracy ,"%")
print("\033[1m--------------------------------------------------------\033[0m")
print("Classification_Report: \n",classification_report(y_test,y_pred_lr))
print("\033[1m--------------------------------------------------------\033[0m")

#### 生成混淆矩阵

混淆矩阵式评估分类模型性能的重要工具，尤其在多分类任务中，能直观呈现模型对各类别样本的预测情况，帮助分析模型的优势与不足。  
其中，横轴为预测标签，纵轴为真实标签。行列交叉处的数值，代表模型将某真实类别的样本预测为另一类别的数量。  
从图中可以看出，除类别3有少量误差，类别4有明显混淆外，其他类别分类准确率都很高。

In [ ]:
# 绘制混淆矩阵
disp = ConfusionMatrixDisplay(confusion_matrix=confusion_matrix(y_test, y_pred_lr), 
                                    display_labels=logreg.classes_)
disp.plot(cmap='Blues')
plt.title('Confusion Matrix')
plt.show()

## 3. 改进

为了改进模型的效果，我们之间增加模型迭代的次数来提升准确率（可能会造成过拟合）。  

In [ ]:
# 导入工具以构建训练集和测试集
from sklearn.model_selection import train_test_split
# 导入必要的包，包括逻辑回归库
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt

# 变量分离：
# X - 特征矩阵：包含所有特征列，但不包含目标列
# Y - 目标向量：只包含Fault_Type列
X = data.drop(['Fault_Type'],axis=1) # 删除目标列作为特征集
y = data['Fault_Type'] # 目标列作为标签集

# 将数据集分成训练集和测试集，这里设置测试集比例为0.2，固定随机种子确保结果可复现
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 逻辑回归模型训练和预测：
# 1. 创建逻辑回归分类器对象，使用默认参数
logreg = LogisticRegression(max_iter=250)
# 2. 使用训练数据拟合模型
logreg.fit(X_train, y_train)
# 3. 使用训练好的模型对测试集进行预测
y_pred_lr = logreg.predict(X_test)

log_train = round(logreg.score(X_train, y_train) * 100, 2)
log_accuracy = round(accuracy_score(y_pred_lr, y_test) * 100, 2)

print("Training Accuracy    :",log_train ,"%")
print("Model Accuracy Score :",log_accuracy ,"%")
print("\033[1m--------------------------------------------------------\033[0m")
print("Classification_Report: \n",classification_report(y_test,y_pred_lr))
print("\033[1m--------------------------------------------------------\033[0m")

# 绘制混淆矩阵
disp = ConfusionMatrixDisplay(confusion_matrix=confusion_matrix(y_test, y_pred_lr), 
                                    display_labels=logreg.classes_)
disp.plot(cmap='Blues')
plt.title('Confusion Matrix')
plt.show()

可以看到，在增加模型迭代次数之后，模型预测的准确率达到了100%。这里可能是出现了过拟合现象。    
可以多次尝试不同的参数。发现在一定范围内随着次数增多，准确率上升。  
一般来说，模型的迭代次数，与模型的参数，数据量应呈正关联。若迭代次数太少，则可能欠拟合，若迭代次数过多，则可能过拟合。

## 4. 总结

Q1：什么是逻辑回归？  
A1：逻辑回归是一种用于解决二分类问题的统计方法，通过Sigmoid函数将线性回归结果映射到(0,1)区间，输出概率值。

Q2：模型的输入输出是什么？  
A2：模型的输入为七千多组数据，每一组数据包含10个变量。可以将变量分为三组，分别是错误类型组变量，电流组变量，电压组变量。输入到模型后，输出错误标签，表示模型认为这组数据属于哪种错误。

Q3：这次的模型有什么缺陷？  
A3：可以看到，在设置模型时并没有输入模型参数。采用的默认模型参数。由于默认的模型迭代轮数较少，可能会导致欠拟合。  
另外，可以看出模型对于少数类的识别能力较差。

Q4：有什么可能的改进方向？  
A4：  
1.解决类别不平衡。可以添加类别权重。  
2.增加模型迭代次数。  
3.模型优化：可以尝试不同的分类器，或者调整模型回归阈值。  
4.针对少量的数据样本，可以收集更多的样本。

Q5：逻辑回归的优势与不足？  
A5：逻辑回归是轻量级分类模型，适合线性问题且需可解释性的场景，但复杂关系需依赖特征工程或换用更复杂的模型。  
**优势**：

1.   **简单高效**：计算量小，训练速度快，适合大规模数据。
2.   **可解释性强**：能直接输出特征权重，便于分析各因素对结果的影响。
3.   **概率输出**：预测结果是概率，可用于排序或设定阈值分类。
4.   **抗噪声**：对轻微的噪声不敏感，且可通过正则化防止过拟合。  

**不足**：  


1.   **线性边界**：只能处理线性可分问题，需结合特征工程拟合非线性关系。
2.   **对异常值敏感**：极端值可能显著影响权重。
3.   **需平衡数据**：若样本类别不平衡（如正负样本比例悬殊），需采用欠采样、过采样或调整权重。




参考资料:  


1.   https://en.wikipedia.org/wiki/Logistic_regression
2.   https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html
3.   LaValley M P. Logistic regression[J]. Circulation, 2008, 117(18): 2395-2399. Available:https://www.ahajournals.org/doi/full/10.1161/CIRCULATIONAHA.106.682658

